In [50]:
# repositories
!git clone https://github.com/Luffy65/Semantic-Correspondence.git # Clone repo
!git clone https://github.com/facebookresearch/dinov3.git # DINOv3
!pip install git+https://github.com/facebookresearch/segment-anything.git # SAM

# Install requirements (requirements.txt)
!pip install -r Semantic-Correspondence/requirements.txt
!pip install -r dinov3/requirements.txt

fatal: destination path 'Semantic-Correspondence' already exists and is not an empty directory.
fatal: destination path 'dinov3' already exists and is not an empty directory.
  Cloning https://github.com/facebookresearch/segment-anything.git to /tmp/pip-req-build-78ha0ctn
  Running command git clone --filter=blob:none --quiet https://github.com/facebookresearch/segment-anything.git /tmp/pip-req-build-78ha0ctn
  Resolved https://github.com/facebookresearch/segment-anything.git to commit dca509fe793f601edb92606367a655c15ac00fdf
  Preparing metadata (setup.py) ... done


In [30]:
# Dependencies
import torch
import os
import shutil
import gzip
import cv2
from google.colab import drive
import matplotlib.pyplot as plt

In [31]:
# Connect google drive, load and unzip data
# 1. Mount Drive
drive.mount('/content/drive')

# 2. Define Paths
DRIVE_ROOT = '/content/drive/MyDrive/AML-PROJECT-DATA/'
DATASET_ROOT = os.path.join(DRIVE_ROOT, 'dataset/')
DATASET_ARCHIVE = os.path.join(DATASET_ROOT, 'SPair-71k.tar.gz')
LOCAL_DATA_DIR = '/content/data'

# 3. Copy and Extract
if not os.path.exists(LOCAL_DATA_DIR):
    print(f"Extracting {DATASET_ARCHIVE} to local VM...")
    os.makedirs(LOCAL_DATA_DIR, exist_ok=True)

    # shutil works for .zip, .tar, .tar.gz, etc.
    # format='gztar' explicitly tells it to handle gzip compression
    shutil.unpack_archive(DATASET_ARCHIVE, LOCAL_DATA_DIR, format='gztar')

    print("Done! Data is ready at:", LOCAL_DATA_DIR)
else:
    print("Data already loaded.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Data already loaded.


In [32]:
# Instantiate models
from segment_anything import SamPredictor, sam_model_registry

DINOV3_REPO_DIR = "dinov3"
CHECKPOINTS_ROOT = os.path.join(DRIVE_ROOT, 'checkpoints/')
SAM_WEIGHTS_PATH = os.path.join(CHECKPOINTS_ROOT, 'sam_vit_h_4b8939.pth')
DINOV3_WEIGHTS_PATH = os.path.join(CHECKPOINTS_ROOT, 'dinov3_vitb16_pretrain_lvd1689m-73cec8be.pth')

sam = sam_model_registry["default"](checkpoint=SAM_WEIGHTS_PATH)

sampredictor = SamPredictor(sam)

dinov2_vitb14 = torch.hub.load('facebookresearch/dinov2', 'dinov2_vitb14')
dinov3_vitb16 = torch.hub.load(DINOV3_REPO_DIR, 'dinov3_vitb16', source='local', weights=DINOV3_WEIGHTS_PATH) # DINOv3 ViT model pretrained on web images

Using cache found in /root/.cache/torch/hub/facebookresearch_dinov2_main


In [33]:
def setup_model_for_finetuning(model, model_type, head, num_layers_to_unfreeze=2):
    """
    Freeze all layers except the last few

    Args:
        model: DINOv2 , DINOv3 or SAM model
        model_type: 'dinov2', 'dinov3' or 'sam'
        num_layers_to_unfreeze: Number of last blocks to unfreeze
        head: if True, unfreeze norm layer for dino , neck for SAM
    """
    # First, freeze everything
    for param in model.parameters():
        param.requires_grad = False

    if 'dinov2' or 'dinov3' in model_type:
        # DINO: Unfreeze last transformer blocks
        total_blocks = len(model.blocks)
        print(f"Total DINO blocks: {total_blocks}")
        print(f"Unfreezing last {num_layers_to_unfreeze} blocks...")

        for i in range(total_blocks - num_layers_to_unfreeze, total_blocks):
            for param in model.blocks[i].parameters():
                param.requires_grad = True

        if head:
            # Also unfreeze the final norm layer
            for param in model.norm.parameters():
                param.requires_grad = True

    elif 'sam' in model_type:
        # SAM: Unfreeze last layers of image encoder
        encoder = model.image_encoder

        if head:
            # Unfreeze neck (final convolutions)
            for param in encoder.neck.parameters():
                param.requires_grad = True

        # Optionally unfreeze last few blocks
        if hasattr(encoder, 'blocks'):
            total_blocks = len(encoder.blocks)
            print(f"Total SAM blocks: {total_blocks}")
            print(f"Unfreezing last {num_layers_to_unfreeze} blocks + neck...")

            for i in range(total_blocks - num_layers_to_unfreeze, total_blocks):
                for param in encoder.blocks[i].parameters():
                    param.requires_grad = True

    # Count trainable parameters
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in model.parameters())

    print(f"Trainable parameters: {trainable_params:,} / {total_params:,} "
          f"({100 * trainable_params / total_params:.2f}%)")

    return model

In [34]:
## Configuration Functions
def createConfig(model_name, config_id, learning_rate=1e-5):
    """
    Creates a configuration dictionary based on model and config ID.

    Config 1: Only norm/neck unfrozen
    Config 2: Norm/neck + last 2 blocks unfrozen

    Args:
        model_name (str): 'dinov2', 'dinov3', or 'sam'
        config_id (int): 1 or 2
        learning_rate (float): Learning rate

    Returns:
        dict: Configuration dictionary
    """
    if config_id == 1:
        # Config 1: Only norm (DINO) or neck (SAM)
        if model_name in ['dinov2', 'dinov3']:
            config = {
                'model_name': model_name,
                'unfreeze_blocks': [],
                'unfreeze_norm': True,
                'unfreeze_neck': False,
                'learning_rate': learning_rate,
                'config_id': 1
            }
        elif model_name == 'sam':
            config = {
                'model_name': model_name,
                'unfreeze_blocks': [],
                'unfreeze_norm': False,
                'unfreeze_neck': True,
                'learning_rate': learning_rate,
                'config_id': 1
            }
        else:
            raise ValueError(f"Unknown model: {model_name}")

    elif config_id == 2:
        # Config 2: Norm/neck + last 2 blocks (10, 11)
        if model_name in ['dinov2', 'dinov3']:
            config = {
                'model_name': model_name,
                'unfreeze_blocks': [10, 11],
                'unfreeze_norm': True,
                'unfreeze_neck': False,
                'learning_rate': learning_rate,
                'config_id': 2
            }
        elif model_name == 'sam':
            config = {
                'model_name': model_name,
                'unfreeze_blocks': [10, 11],
                'unfreeze_norm': False,
                'unfreeze_neck': True,
                'learning_rate': learning_rate,
                'config_id': 2
            }
        else:
            raise ValueError(f"Unknown model: {model_name}")
    else:
        raise ValueError("config_id must be 1 or 2")

    return config

def applyConfig(model, config):
    """
    Applies the configuration to freeze/unfreeze layers in the model.

    Args:
        model: PyTorch model (DINOv2, DINOv3, or SAM)
        config (dict): Configuration dictionary from createConfig()

    Returns:
        int: Number of trainable parameters
    """
    model_name = config['model_name']
    unfreeze_blocks = config['unfreeze_blocks']
    unfreeze_norm = config['unfreeze_norm']
    unfreeze_neck = config['unfreeze_neck']

    # First, freeze all parameters
    for param in model.parameters():
        param.requires_grad = False

    print(f"\n{'='*60}")
    print(f"Applying Configuration {config['config_id']} for {model_name.upper()}")
    print(f"{'='*60}")

    unfrozen_params = 0

    if model_name in ['dinov2', 'dinov3']:
        # DINOv2/v3 Architecture:
        # - model.blocks[0-11]: Transformer blocks
        # - model.norm: Final LayerNorm

        if len(unfreeze_blocks) > 0:
            print(f"Unfreezing blocks: {unfreeze_blocks}")

            for block_idx in unfreeze_blocks:
                layer_name = f"blocks.{block_idx}"
                for name, param in model.named_parameters():
                    if layer_name in name:
                        param.requires_grad = True
                        unfrozen_params += param.numel()
                print(f"  - Block {block_idx}")

        if unfreeze_norm:
            print("Unfreezing normalization layers:")
            for name, param in model.named_parameters():
                if 'norm' in name:
                    param.requires_grad = True
                    unfrozen_params += param.numel()
                    print(f"  - {name}")

    elif model_name == 'sam':
        # SAM Image Encoder Architecture (ViT):
        # - model.image_encoder.blocks[0-11]: Transformer blocks
        # - model.image_encoder.neck: Neck layers (conv layers)

        if len(unfreeze_blocks) > 0:
            print(f"Unfreezing blocks: {unfreeze_blocks}")

            for block_idx in unfreeze_blocks:
                layer_name = f"image_encoder.blocks.{block_idx}"
                for name, param in model.named_parameters():
                    if layer_name in name:
                        param.requires_grad = True
                        unfrozen_params += param.numel()
                print(f"  - Block {block_idx}")

        if unfreeze_neck:
            print("Unfreezing neck layers:")
            for name, param in model.named_parameters():
                if 'image_encoder.neck' in name:
                    param.requires_grad = True
                    unfrozen_params += param.numel()
                    print(f"  - {name}")

    # Count total parameters
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

    print(f"\n{'='*60}")
    print(f"Parameter Summary:")
    print(f"  Total parameters:     {total_params:,}")
    print(f"  Trainable parameters: {trainable_params:,} ({100*trainable_params/total_params:.2f}%)")
    print(f"  Frozen parameters:    {total_params-trainable_params:,} ({100*(total_params-trainable_params)/total_params:.2f}%)")
    print(f"{'='*60}\n")

    return trainable_params

## Training Functions (Placeholder)
import torch.optim as optim
from torch.utils.data import DataLoader
from tqdm import tqdm

def keypointLoss(pred_coords, target_coords, valid_mask=None):
    """
    Computes L2 loss between predicted and target keypoint coordinates.

    Args:
        pred_coords: (B, N, 2) predicted (x, y) coordinates
        target_coords: (B, N, 2) target (x, y) coordinates
        valid_mask: (B, N) boolean mask for valid keypoints

    Returns:
        loss: scalar loss value
    """
    diff = pred_coords - target_coords
    loss = torch.sqrt((diff ** 2).sum(dim=-1) + 1e-8)  # L2 distance

    if valid_mask is not None:
        loss = loss * valid_mask
        loss = loss.sum() / (valid_mask.sum() + 1e-8)
    else:
        loss = loss.mean()

    return loss


def trainEpoch(model, dataloader, optimizer, device, config):
    """
    Trains the model for one epoch.

    TODO: Implement forward pass and keypoint matching

    Args:
        model: PyTorch model
        dataloader: Training data loader
        optimizer: Optimizer
        device: Device to train on
        config: Configuration dictionary

    Returns:
        avg_loss: Average loss for the epoch
    """
    model.train()
    total_loss = 0.0
    num_batches = 0

    pbar = tqdm(dataloader, desc="Training")
    for batch in pbar:
        src_imgs = batch['src_img'].to(device)
        trg_imgs = batch['trg_img'].to(device)
        src_kps = batch['src_kps'].to(device)
        trg_kps = batch['trg_kps'].to(device)
        valid_mask = batch['valid_mask'].to(device)

        optimizer.zero_grad()

        # TODO: Implement forward pass and keypoint matching
        # pred_kps = model.match_keypoints(src_imgs, trg_imgs, src_kps)
        # loss = keypointLoss(pred_kps, trg_kps, valid_mask)

        # Placeholder loss
        loss = torch.tensor(0.0, requires_grad=True, device=device)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        num_batches += 1

        pbar.set_postfix({'loss': f'{loss.item():.4f}'})

    return total_loss / num_batches if num_batches > 0 else 0.0
## Run Training



Scegli il modello e la configurazione:
- **model**: 'dinov2', 'dinov3', 'sam'
- **config**: 1 (solo norm/neck), 2 (norm/neck + ultimi 2 blocchi)

In [87]:
# # --- 3. Training Logic ---

def extract_descriptors(model, img, model_type, patch_size):
    """
    Extract dense features from the model
    Args:
        model: DINO or SAM model
        img: Input image tensor (B, 3, H, W) - already normalized by dataset
        model_type: 'dinov2' or 'sam'
        patch_size: Patch size of the model
    Returns:
        features: (B, H_feat, W_feat, C)
    """
    if 'dinov2' in model_type or 'dinov3' in model_type:
        res = model.forward_features(img)
        x = res['x_norm_patchtokens']
        B, N, C = x.shape
        H = img.shape[2] // patch_size
        W = img.shape[3] // patch_size
        x = x.reshape(B, H, W, C)
        return x

    elif 'sam' in model_type:
        # SAM expects images in range [0, 1], but your dataset normalizes to ImageNet stats
        # You may need to denormalize first if SAM doesn't work with normalized images
        x = model.image_encoder(img)
        x = x.permute(0, 2, 3, 1)  # (B, C, H, W) -> (B, H, W, C)
        return x

    return None

def train_epoch(model, train_loader, optimizer, device, model_type, patch_size):
    """
    Train for one epoch using cross-entropy loss on similarity maps
    """
    model.train()
    total_loss = 0
    num_batches = 0

    for batch_idx, batch in enumerate(train_loader):
        src_img = batch['src_img'].to(device)  # (B, 3, H, W) Already normalized
        trg_img = batch['trg_img'].to(device)
        src_kps_list = batch['src_kps']  # List[Tensor(N_i, 2)] <-- MODIFICATO
        trg_kps_list = batch['trg_kps']  # List[Tensor(N_i, 2)] <-- MODIFICATO

        optimizer.zero_grad()

        # Extract dense features
        src_feat = extract_descriptors(model, src_img, model_type, patch_size)
        trg_feat = extract_descriptors(model, trg_img, model_type, patch_size)

        B, Hf, Wf, C = src_feat.shape
        H_img, W_img = src_img.shape[2], src_img.shape[3]  # Current image size (224x224)

        # Normalize features for cosine similarity
        src_feat = F.normalize(src_feat, dim=-1)
        trg_feat = F.normalize(trg_feat, dim=-1)

        # Compute loss for all keypoints in the batch
        batch_loss = 0
        num_valid_kps = 0

        for b in range(B):
            # Prendi i keypoint per questa immagine (ora sono tensori singoli, non slice)
            src_kps = src_kps_list[b].to(device)  # (N_i, 2) <-- MODIFICATO
            trg_kps = trg_kps_list[b].to(device)  # (N_i, 2) <-- MODIFICATO

            num_kps = src_kps.shape[0]  # Numero di keypoint per questa coppia

            for kp_idx in range(num_kps):  # <-- MODIFICATO (era src_kps.shape[1])
                src_kp = src_kps[kp_idx]  # <-- MODIFICATO (era src_kps[b, kp_idx])
                trg_kp = trg_kps[kp_idx]  # <-- MODIFICATO

                # Skip invalid keypoints (check for NaN or negative values)
                if torch.isnan(src_kp).any() or torch.isnan(trg_kp).any():
                    continue
                if src_kp[0] < 0 or src_kp[1] < 0 or trg_kp[0] < 0 or trg_kp[1] < 0:
                    continue

                # Map source keypoint to feature grid
                feat_x = int((src_kp[0] / W_img) * Wf)
                feat_y = int((src_kp[1] / H_img) * Hf)
                feat_x = max(0, min(feat_x, Wf - 1))
                feat_y = max(0, min(feat_y, Hf - 1))

                # Extract source descriptor
                src_desc = src_feat[b, feat_y, feat_x, :]  # (C,)

                # Compute similarity with ALL target locations
                trg_feat_flat = trg_feat[b].reshape(Hf * Wf, C)  # (H*W, C)
                sim = torch.matmul(trg_feat_flat, src_desc)  # (H*W,)

                # Ground truth: target keypoint location in feature grid
                trg_feat_x = int((trg_kp[0] / W_img) * Wf)
                trg_feat_y = int((trg_kp[1] / H_img) * Hf)
                trg_feat_x = max(0, min(trg_feat_x, Wf - 1))
                trg_feat_y = max(0, min(trg_feat_y, Hf - 1))

                gt_idx = trg_feat_y * Wf + trg_feat_x

                # Cross-entropy loss
                kp_loss = F.cross_entropy(
                    sim.unsqueeze(0),
                    torch.tensor([gt_idx], device=device)
                )

                batch_loss += kp_loss
                num_valid_kps += 1

        # Average loss over valid keypoints
        if num_valid_kps > 0:
            batch_loss = batch_loss / num_valid_kps
            batch_loss.backward()
            optimizer.step()

            total_loss += batch_loss.item()
            num_batches += 1

        if batch_idx % 10 == 0:
            print(f"  Batch {batch_idx}/{len(train_loader)} | Loss: {batch_loss.item():.4f} | Valid KPs: {num_valid_kps}")

    avg_loss = total_loss / num_batches if num_batches > 0 else 0
    return avg_loss


def validate_epoch(model, val_loader, device, model_type, patch_size, thresholds=[0.05, 0.1, 0.2]):
    """
    Compute PCK on validation set using the precomputed pck_threshold from dataset
    """
    model.eval()

    correct_kps = {t: 0 for t in thresholds}
    total_kps = 0

    with torch.no_grad():
        for batch in val_loader:
            src_img = batch['src_img'].to(device)
            trg_img = batch['trg_img'].to(device)
            src_kps = batch['src_kps']
            trg_kps = batch['trg_kps']
            trg_bbox = batch['trg_bbox']  # (B, 4)

            # Extract features
            src_feat = extract_descriptors(model, src_img, model_type, patch_size)
            trg_feat = extract_descriptors(model, trg_img, model_type, patch_size)

            B, Hf, Wf, C = src_feat.shape
            H_img, W_img = src_img.shape[2], src_img.shape[3]

            # Normalize
            src_feat = F.normalize(src_feat, dim=-1)
            trg_feat = F.normalize(trg_feat, dim=-1)

            for b in range(B):
                # PCK normalization factor
                bbox = trg_bbox[b]
                bbox_w = bbox[2] - bbox[0]
                bbox_h = bbox[3] - bbox[1]
                norm_factor = max(bbox_w, bbox_h).item()

                for kp_idx in range(src_kps.shape[1]):
                    src_kp = src_kps[b, kp_idx]
                    trg_kp = trg_kps[b, kp_idx]

                    # Skip invalid keypoints
                    if torch.isnan(src_kp).any() or torch.isnan(trg_kp).any():
                        continue
                    if src_kp[0] < 0 or src_kp[1] < 0 or trg_kp[0] < 0 or trg_kp[1] < 0:
                        continue

                    # Map to feature grid
                    feat_x = int((src_kp[0] / W_img) * Wf)
                    feat_y = int((src_kp[1] / H_img) * Hf)
                    feat_x = max(0, min(feat_x, Wf - 1))
                    feat_y = max(0, min(feat_y, Hf - 1))

                    # Get source descriptor
                    src_desc = src_feat[b, feat_y, feat_x, :]

                    # Compute similarity map
                    trg_feat_b = trg_feat[b].reshape(Hf * Wf, C)
                    sim = torch.matmul(trg_feat_b, src_desc)

                    # Find best match
                    best_idx = torch.argmax(sim)
                    pred_y = (best_idx // Wf).item()
                    pred_x = (best_idx % Wf).item()

                    # Map back to pixels
                    pred_x_pix = (pred_x / Wf) * W_img
                    pred_y_pix = (pred_y / Hf) * H_img

                    # Compute distance
                    dist = np.sqrt((pred_x_pix - trg_kp[0].item())**2 +
                                 (pred_y_pix - trg_kp[1].item())**2)

                    total_kps += 1
                    for t in thresholds:
                        if dist <= (t * norm_factor):
                            correct_kps[t] += 1

    pck_scores = {t: (correct_kps[t] / total_kps) if total_kps > 0 else 0.0
                  for t in thresholds}

    return pck_scores

In [86]:
from torchvision.transforms import v2

def train_light_finetuning(
    model,
    pair_ann_path,
    layout_path,
    image_path,
    model_type,
    patch_size,
    head,
    dataset_size='large',
    pck_alpha=0.1,
    num_layers_to_unfreeze=3,
    num_epochs=5,
    batch_size=8,
    learning_rate=5e-5,
    device='cuda'

):
    """
    Main training loop for light fine-tuning
    """
    print(f"\n{'='*70}")
    print(f"Starting Light Fine-tuning")
    print(f"{'='*70}")
    print(f"Model: {model_type}")
    print(f"Layers to unfreeze: {num_layers_to_unfreeze}")
    print(f"Epochs: {num_epochs}")
    print(f"Batch size: {batch_size}")
    print(f"Learning rate: {learning_rate}")
    print(f"{'='*70}\n")

    # Setup model
    model = setup_model_for_finetuning(model, model_type, num_layers_to_unfreeze, head)
    model = model.to(device)

    # Create datasets using YOUR existing class
    train_dataset = SPairDataset(
        pair_ann_path, layout_path, image_path,
        dataset_size, pck_alpha, datatype='trn'
    )
    val_dataset = SPairDataset(
        pair_ann_path, layout_path, image_path,
        dataset_size, pck_alpha, datatype='val'
    )

    # transforms = v2.Compose([
    #     v2.Resize((224, 224)),
    #     v2.ToDtype(torch.float32, scale=True),
    # ])

    # for img in train_dataset:
    #     img = transforms(img)

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=0,
        collate_fn=custom_collate_fn
    )

    # for img in val_dataset:
    #     img = transforms(img)

    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
        collate_fn=custom_collate_fn
    )

    print(f"Train samples: {len(train_dataset)}")
    print(f"Val samples: {len(val_dataset)}\n")

    # Optimizer (only for trainable parameters)
    optimizer = torch.optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=learning_rate
    )

    # Training loop
    best_pck = 0.0
    history = {
        'train_loss': [],
        'val_pck_005': [],
        'val_pck_010': [],
        'val_pck_020': []
    }

    for epoch in range(num_epochs):
        print(f"\n{'='*70}")
        print(f"Epoch {epoch + 1}/{num_epochs}")
        print(f"{'='*70}")

        # Train
        train_loss = train_epoch(
            model, train_loader, optimizer, device, model_type, patch_size
        )

        # Validate
        val_pck = validate_epoch(
            model, val_loader, device, model_type, patch_size
        )

        # Record history
        history['train_loss'].append(train_loss)
        history['val_pck_005'].append(val_pck[0.05])
        history['val_pck_010'].append(val_pck[0.1])
        history['val_pck_020'].append(val_pck[0.2])

        # Print results
        print(f"\n{'─'*70}")
        print(f"Epoch {epoch + 1} Results:")
        print(f"  Train Loss:   {train_loss:.4f}")
        print(f"  Val PCK@0.05: {val_pck[0.05]:.3f}")
        print(f"  Val PCK@0.10: {val_pck[0.1]:.3f}")
        print(f"  Val PCK@0.20: {val_pck[0.2]:.3f}")

        # Save best model
        if val_pck[0.1] > best_pck:
            best_pck = val_pck[0.1]
            save_path = f'best_model_{model_type}_layers{num_layers_to_unfreeze}.pth'
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'pck': val_pck,
                'history': history
            }, save_path)
            print(f"  ✓ New best model saved! ({save_path})")

        print(f"{'─'*70}")

    print(f"\n{'='*70}")
    print(f"Training Complete!")
    print(f"Best Val PCK@0.1: {best_pck:.3f}")
    print(f"{'='*70}\n")

    return history

In [37]:
def plot_training_history(history):
    """
    Plot training loss and validation PCK
    """
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

    # Loss
    ax1.plot(history['train_loss'], marker='o')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Training Loss')
    ax1.set_title('Training Loss over Epochs')
    ax1.grid(True)

    # PCK
    ax2.plot(history['val_pck_005'], marker='o', label='PCK@0.05')
    ax2.plot(history['val_pck_010'], marker='s', label='PCK@0.10')
    ax2.plot(history['val_pck_020'], marker='^', label='PCK@0.20')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('PCK Score')
    ax2.set_title('Validation PCK over Epochs')
    ax2.legend()
    ax2.grid(True)

    plt.tight_layout()
    plt.savefig('training_history.png', dpi=150)
    plt.show()

In [38]:
print(os.getcwd())

/content


In [39]:
import argparse
import sys
sys.path.append('/content/Semantic-Correspondence/datasets/')
from SPairDataset import SPairDataset
import torch

def main(args):

  parser = argparse.ArgumentParser()
  parser.add_argument('--model', type=str, default='dinov2_vitb14', choices=['dinov2_vitb14', 'sam_vit_h', 'dinov3_vitb16'])
  # parser.add_argument('--config', type=int, default=1, choices=[1, 2], help='1: Norm+LastBlock, 2: Norm+Last2Blocks')
  # Aggiungi epochs agli argomenti o usa il default definito sotto
  parser.add_argument('--head', type=bool, default=True, choices=[True, False], help='True: unfreeze neck or norm, False otherwise')

  parser.add_argument('--epochs', type=int, default=10)

  # Parse only known arguments to avoid issues with Jupyter/Colab kernel arguments
  if 'ipykernel' in sys.modules:
      args = parser.parse_args([]) # Pass an empty list for notebook execution
  else:
      args = parser.parse_args() # For command-line execution

  head = args.head
  # Paths (use your existing setup)
  # script_dir = os.getcwd()
  # The dataset is extracted to /content/data, not directly /content
  data_root = '/content/data/'
  relative_path = os.path.join(data_root, "SPair-71k") + os.sep
  pair_ann_path = relative_path + 'PairAnnotation'
  layout_path = relative_path + 'Layout'
  image_path = relative_path + 'JPEGImages'

  # Load pretrained DINO model
  model = ''
  # --- Model Loading 'Logic ---
  # NOTA: Assicurati che le variabili globali (dinov2_vitb14, ecc.) esistano
  # se stai eseguendo in un notebook, altrimenti usa torch.hub.load qui dentro.
  model_type = args.model
  if 'dinov2' in model_type:
      # Se non sono definite globalmente, caricale qui:
      # model = torch.hub.load('facebookresearch/dinov2', model_name)
      model = dinov2_vitb14
  elif 'dinov3' in model_type:
      model = dinov3_vitb16
  elif 'sam' in model_type:
      model = sam
  else:
      raise ValueError(f"Unknown model: {model_type}")
  patch_size = 14

  device = 'cuda' if torch.cuda.is_available() else 'cpu'

  # Train with different configurations


  history = train_light_finetuning(
      model=model,
      pair_ann_path=pair_ann_path,
      layout_path=layout_path,
      image_path=image_path,
      model_type=model_type,
      patch_size=patch_size,
      head=head,
      dataset_size='large',
      pck_alpha=0.1,
      num_epochs=20,
      batch_size=8,
      learning_rate=1e-5,
      device=device)

  # Plot results
  plot_training_history(history)

    # # --- Setup Dataset Paths ---
    # # Ottieni il percorso assoluto della cartella dove si trova questo script
    # script_dir = os.path.dirname(os.path.abspath(__file__))

    # # Costruisci i path relativi
    # spair_path = os.path.join(script_dir, "SPair-71k")
    # pair_ann_path = os.path.join(spair_path, 'PairAnnotation')
    # layout_path = os.path.join(spair_path, 'Layout')
    # image_path = os.path.join(spair_path, 'JPEGImages')

    # # --- Setup Output Directory ---
    # # Definisci la cartella checkpoints dentro la repository corrente
    # output_dir = os.path.join(script_dir, "checkpoints")
    # os.makedirs(output_dir, exist_ok=True) # Crea se non esiste

    # dataset_size = 'large'
    # pck_alpha = 0.1

    # trn_dataset = SPairDataset(pair_ann_path, layout_path, image_path, dataset_size, pck_alpha, datatype='trn')
    # # Nota: batch_size deve essere passato al DataLoader, non definito solo come variabile
    # batch_size = 8
    # trn_dataloader = DataLoader(trn_dataset, batch_size=batch_size, shuffle=True, num_workers=2)

    # learning_rate = 1e-5
    # epochs = args.epochs # Usa quello da argparse
    # patch_size = 16
    # device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    # print(f"Device: {device}\n")

    # model_name = args.model
    # config_id = args.config



    # model = model.to(device)

    # # Crea e applica configurazione
    # # Assumo che createConfig e applyConfig siano definite altrove nel tuo codice
    # config = createConfig(model_name, config_id, learning_rate)
    # trainable_params_count = applyConfig(model, config) # Modificato per catturare il return

    # # Setup optimizer
    # optimizer = optim.AdamW(
    #     filter(lambda p: p.requires_grad, model.parameters()),
    #     lr=config['learning_rate'],
    #     weight_decay=1e-4
    # )

    # print(f"\n{'='*60}")
    # print("READY FOR TRAINING")
    # print(f"{'='*60}")
    # print(f"Model: {config['model_name']}")
    # print(f"Config ID: {config['config_id']}")
    # print(f"Trainable params: {trainable_params_count:,}")
    # print(f"Learning rate: {config['learning_rate']}")
    # print(f"Epochs: {epochs}")
    # print(f"Batch size: {batch_size}")
    # print(f"Output Checkpoints Dir: {output_dir}")
    # print(f"{'='*60}\n")




In [85]:
import argparse
import sys
sys.path.append('/content/Semantic-Correspondence/datasets/')
from SPairDataset import SPairDataset
import torch

def main_alt(model, head, epoch):

  model_type = model
  head = head
  epoch = epoch

  # Paths (use your existing setup)
  # script_dir = os.getcwd()
  # The dataset is extracted to /content/data, not directly /content
  data_root = '/content/data/'
  relative_path = os.path.join(data_root, "SPair-71k") + os.sep
  pair_ann_path = relative_path + 'PairAnnotation'
  layout_path = relative_path + 'Layout'
  image_path = relative_path + 'JPEGImages'

  # Load pretrained DINO model

  # --- Model Loading 'Logic ---
  # NOTA: Assicurati che le variabili globali (dinov2_vitb14, ecc.) esistano
  # se stai eseguendo in un notebook, altrimenti usa torch.hub.load qui dentro.
  model_type = model
  if 'dinov2' in model_type:
      # Se non sono definite globalmente, caricale qui:
      # model = torch.hub.load('facebookresearch/dinov2', model_name)
      model = dinov2_vitb14
  elif 'dinov3' in model_type:
      model = dinov3_vitb16
  elif 'sam' in model_type:
      model = sam
  else:
      raise ValueError(f"Unknown model: {model_type}")
  patch_size = 16

  device = 'cuda' if torch.cuda.is_available() else 'cpu'

  # Train with different configurations


  history = train_light_finetuning(
      model=model,
      pair_ann_path=pair_ann_path,
      layout_path=layout_path,
      image_path=image_path,
      model_type=model_type,
      patch_size=patch_size,
      head=head,
      dataset_size='large',
      pck_alpha=0.1,
      num_epochs=5,
      batch_size=8,
      learning_rate=5e-5,
      device=device)

  # Plot results
  plot_training_history(history)


In [93]:
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset
import json
import os
from PIL import Image
import numpy as np

def read_img(path):
    img = np.array(Image.open(path).convert('RGB'))
    return torch.tensor(img.transpose(2, 0, 1).astype(np.float32))

class SPairDataset(Dataset):
    def __init__(self, pair_ann_path, layout_path, image_path, dataset_size, pck_alpha, datatype):
        self.datatype = datatype
        self.pck_alpha = pck_alpha
        ann_list = os.path.join(layout_path, dataset_size, datatype + '.txt')
        self.ann_files = [x.strip() for x in open(ann_list, "r").readlines() if x.strip()]
        self.pair_ann_path = pair_ann_path
        self.image_path = image_path

    def __len__(self):
        return len(self.ann_files)

    def __getitem__(self, idx):
      ann_file = self.ann_files[idx] + '.json'
      with open(os.path.join(self.pair_ann_path, self.datatype, ann_file)) as f:
          annotation = json.load(f)

      category = annotation['category']
      src_img = read_img(os.path.join(self.image_path, category, annotation['src_imname']))
      trg_img = read_img(os.path.join(self.image_path, category, annotation['trg_imname']))

      # ========== RESIZE ==========
      TARGET_H, TARGET_W = 224, 224

      # Salva dimensioni originali
      orig_h_src, orig_w_src = src_img.shape[1], src_img.shape[2]
      orig_h_trg, orig_w_trg = trg_img.shape[1], trg_img.shape[2]

      # Calcola fattori di scala
      scale_x_src = TARGET_W / orig_w_src
      scale_y_src = TARGET_H / orig_h_src
      scale_x_trg = TARGET_W / orig_w_trg
      scale_y_trg = TARGET_H / orig_h_trg

      # Resize immagini
      src_img = F.interpolate(
          src_img.unsqueeze(0),
          size=(TARGET_H, TARGET_W),
          mode='bilinear',
          align_corners=False
      ).squeeze(0).clone()

      trg_img = F.interpolate(
          trg_img.unsqueeze(0),
          size=(TARGET_H, TARGET_W),
          mode='bilinear',
          align_corners=False
      ).squeeze(0).clone()

      # Normalizza (0-255 -> 0-1 -> ImageNet normalization)
      mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
      std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
      src_img = (src_img / 255.0 - mean) / std
      trg_img = (trg_img / 255.0 - mean) / std

      # Scala keypoint
      src_kps = torch.tensor(annotation['src_kps'], dtype=torch.float32)
      src_kps[:, 0] *= scale_x_src
      src_kps[:, 1] *= scale_y_src

      trg_kps = torch.tensor(annotation['trg_kps'], dtype=torch.float32)
      trg_kps[:, 0] *= scale_x_trg
      trg_kps[:, 1] *= scale_y_trg

      # ========== BBOX E PCK THRESHOLD ==========
      trg_bbox = annotation['trg_bndbox']  # [xmin, ymin, xmax, ymax]

      # Scala il bbox alle nuove dimensioni (coerente con immagine ridimensionata)
      trg_bbox_scaled = [
          trg_bbox[0] * scale_x_trg,
          trg_bbox[1] * scale_y_trg,
          trg_bbox[2] * scale_x_trg,
          trg_bbox[3] * scale_y_trg
      ]

      # Calcola threshold PCK sul bbox scalato
      pck_threshold = max(
          trg_bbox_scaled[2] - trg_bbox_scaled[0],
          trg_bbox_scaled[3] - trg_bbox_scaled[1]
      ) * self.pck_alpha

      return {
          'src_img': src_img,              # (3, 224, 224)
          'trg_img': trg_img,              # (3, 224, 224)
          'src_kps': src_kps,              # (N, 2) scalati
          'trg_kps': trg_kps,              # (N, 2) scalati
          'trg_bbox': torch.tensor(trg_bbox_scaled, dtype=torch.float32),  # (4,)
          'pck_threshold': pck_threshold,   # Scalare float
          'category': annotation.get('category', ''),
          'src_imsize': (orig_h_src, orig_w_src),  # Dimensioni originali
          'trg_imsize': (orig_h_trg, orig_w_trg)
      }



# Test rapido
print("Classe SPairDataset ridefinita. Testando...")


Classe SPairDataset ridefinita. Testando...


In [88]:
def custom_collate_fn(batch):
    """
    Gestisce batch con numero variabile di keypoint.
    """
    src_imgs = torch.stack([item['src_img'] for item in batch])
    trg_imgs = torch.stack([item['trg_img'] for item in batch])

    # Keypoint: lista (numero variabile)
    src_kps = [item['src_kps'] for item in batch]
    trg_kps = [item['trg_kps'] for item in batch]

    # Bbox e threshold: possono essere stackati (dimensione fissa)
    trg_bbox = torch.stack([item['trg_bbox'] for item in batch])  # (B, 4)
    pck_threshold = torch.tensor([item['pck_threshold'] for item in batch])  # (B,)

    categories = [item['category'] for item in batch]

    return {
        'src_img': src_imgs,
        'trg_img': trg_imgs,
        'src_kps': src_kps,
        'trg_kps': trg_kps,
        'trg_bbox': trg_bbox,           # <-- AGGIUNTO
        'pck_threshold': pck_threshold,  # <-- AGGIUNTO
        'category': categories
    }



In [ ]:

if __name__ == '__main__':
    main_alt('dinov3', True, 7)


Starting Light Fine-tuning
Model: dinov3
Layers to unfreeze: 3
Epochs: 5
Batch size: 8
Learning rate: 5e-05

Total DINO blocks: 12
Unfreezing last True blocks...
Trainable parameters: 7,090,944 / 85,669,632 (8.28%)
Train samples: 53340
Val samples: 5384


Epoch 1/5
  Batch 0/6668 | Loss: 3.7774 | Valid KPs: 54
  Batch 10/6668 | Loss: 3.8518 | Valid KPs: 46
  Batch 20/6668 | Loss: 3.8612 | Valid KPs: 57
  Batch 30/6668 | Loss: 3.8543 | Valid KPs: 57
  Batch 40/6668 | Loss: 3.9543 | Valid KPs: 72
  Batch 50/6668 | Loss: 3.8622 | Valid KPs: 65
  Batch 60/6668 | Loss: 3.8706 | Valid KPs: 45
  Batch 70/6668 | Loss: 3.7486 | Valid KPs: 43
  Batch 80/6668 | Loss: 3.7763 | Valid KPs: 51
  Batch 90/6668 | Loss: 3.7129 | Valid KPs: 60
  Batch 100/6668 | Loss: 3.8075 | Valid KPs: 58
  Batch 110/6668 | Loss: 3.8493 | Valid KPs: 47
  Batch 120/6668 | Loss: 3.9016 | Valid KPs: 57
  Batch 130/6668 | Loss: 3.8293 | Valid KPs: 68
  Batch 140/6668 | Loss: 3.7554 | Valid KPs: 43
  Batch 150/6668 | Loss:

In [ ]:
!python train.py --model dinov2 --head True

In [ ]:
%%writefile train.py

In [ ]:
!ls -l

In [52]:
import inspect
print(inspect.getsource(SPairDataset.__getitem__))


        def __getitem__(self, idx):
          # ... (caricamento file json e path come prima) ...
          # ...
          src_img = read_img(...) # Tensore C, H, W
          trg_img = read_img(...)

          # --- PUNTO DI INSERIMENTO: RESIZE MANUALE ---
          # Definisci la dimensione target (es. passata in __init__ o fissa qui)
          target_h, target_w = 224, 224 

          # 1. Calcola fattori di scala
          orig_h_src, orig_w_src = src_img.shape[1], src_img.shape[2]
          orig_h_trg, orig_w_trg = trg_img.shape[1], trg_img.shape[2]
          
          scale_x_src = target_w / orig_w_src
          scale_y_src = target_h / orig_h_src
          scale_x_trg = target_w / orig_w_trg
          scale_y_trg = target_h / orig_h_trg

          # 2. Ridimensiona le immagini (Interpolazione)
          # src_img è (C, H, W). F.interpolate vuole (B, C, H, W).
          import torch.nn.functional as F
          src_img = F.interpolate(src_img.unsqueeze(0), size=(target_h, targe